# ND2 Unified Discovery Notebook: Fase 3 (Legendre Generalizado)

Objetivo: Descubrir la ley universal $V \propto \frac{P_l(\cos \theta)}{r^{l+1}}$ para cualquier orden $l$.

In [ ]:
# 0. Sincronización Robusta (Colab/Kaggle)
import os

# Detectar entorno y definir rutas
is_colab = os.path.exists('/content')
base_path = '/content' if is_colab else '/kaggle/working'
target_dir = os.path.join(base_path, 'ND2')

# Clonar si no existe
if not os.path.exists(target_dir):
    print(f"Clonando repositorio en {target_dir}...")
    repo_url = "https://github.com/manramirezpi/ND2.git"
    os.system(f"git clone {repo_url} {target_dir}")

# Cambiar de directorio y actualizar
os.chdir(target_dir)
os.system("git pull origin main")
print(f"\n[OK] Trabajando en: {os.getcwd()}")

In [ ]:
# 1. Instalación de Dependencias
!pip install gplearn setproctitle geopandas
!mkdir -p weights Multipolos/data/nd2_format Multipolos/results
if not os.path.exists('weights/checkpoint.pth'):
    !wget -O weights/checkpoint.pth https://github.com/yuzhTHU/ND2/releases/download/checkpoint.pth/checkpoint.pth

In [ ]:
# 2. GENERAR DATASET MULTI-L (Fase 3)
import numpy as np, json, os
def generate_multi_l_data(num_samples_per_l=500):
    all_r, all_theta, all_l, all_V = [], [], [], []
    
    for l in [1, 2, 3]:
        r = np.random.uniform(0.5, 6.0, num_samples_per_l)
        theta = np.random.uniform(0, np.pi, num_samples_per_l)
        
        if l == 1: # Dipolo
            P = np.cos(theta)
        elif l == 2: # Cuadrupolo
            P = 0.5 * (3 * np.cos(theta)**2 - 1)
        elif l == 3: # Octupolo
            P = 0.5 * (5 * np.cos(theta)**3 - 3 * np.cos(theta))
            
        V = P / (r**(l+1))
        
        all_r.extend(r)
        all_theta.extend(theta)
        all_l.extend([float(l)] * num_samples_per_l)
        all_V.extend(V)

    num_total = len(all_V)
    data = {"V": 2, "E": 1, "A": [[0,1],[0,0]], "G": [[0,1]],
            "l_order": [[0.0, float(li)] for li in all_l],
            "theta": [[0.0, float(ti)] for ti in all_theta],
            "r_node": [[0.0, float(ri)] for ri in all_r],
            "target": [[0.0, float(vi)] for vi in all_V]}
    
    os.makedirs("Multipolos/data/nd2_format", exist_ok=True)
    json.dump(data, open("Multipolos/data/nd2_format/MULTIL_nd2.json", "w"))
    print(f"Dataset Multi-L generado ({num_total} muestras).")

generate_multi_l_data()

In [ ]:
# 3. EJECUTAR BÚSQUEDA GENERAL (Fase 3)
!python3 search.py \
    --data Multipolos/data/nd2_format/MULTIL_nd2.json \
    --vars l_order theta r_node \
    --target_var target \
    --episodes 1000 \
    --device cuda

In [ ]:
# 4. AUTO-PUSH INDEPENDIENTE
def get_token_reloaded():
    try: from kaggle_secrets import UserSecretsClient; return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except: pass
    try: from google.colab import userdata; return userdata.get('GITHUB_TOKEN')
    except: pass
    return None

token = get_token_reloaded()
if token:
    print("Sincronizando con GitHub...")
    !git config --global user.email "manuel@researcher.phys"
    !git config --global user.name "ND2-Auto-Agent"
    !git remote set-url origin https://{token}@github.com/manramirezpi/ND2.git
    !git add Multipolos/results/ Multipolos/data/nd2_format/MULTIL_nd2.json
    !git commit -m "Fase 3: Resultados de Búsqueda Generalizada (Multi-L)"
    !git push origin main
    print("\n[OK] Ciclo cerrado. Avísale al Agente Antigravity para el análisis final.")
else:
    print("\n[!] GITHUB_TOKEN no encontrado.")